In [ ]:
import pandas as pd
from pathlib import Path

RUTA_provisional = Path("../data/provisional")

carpeta_resenas = (
    RUTA_provisional / "resenas_videojuegos_bloques"
)

ruta_metadatos = (
    RUTA_provisional / "metadatos_videojuegos.parquet"
)

archivos_resenas = sorted(
    carpeta_resenas.glob("*.parquet")
)

print("Archivos de reseñas encontrados:", len(archivos_resenas))
print("Archivo de metadatos encontrado:", ruta_metadatos.exists())

Archivos de reseñas encontrados: 47
Archivo de metadatos encontrado: True


In [15]:
# comprobación de la existencia de reseñas
total_resenas = 0

for archivo in archivos_resenas:
    bloque = pd.read_parquet(
        archivo,
        columns=["parent_asin"]
    )
    total_resenas += len(bloque)

print("Total de reseñas:", f"{total_resenas:,}")

Total de reseñas: 1,502,718


In [4]:
# comprobación de la existencia de videojuegos con reseñas
ids_prod_res = set()

for archivo in archivos_resenas:
    bloque = pd.read_parquet(
        archivo,
        columns=["parent_asin"]
    )

    ids_prod_res.update(
        bloque["parent_asin"]
        .dropna()
        .astype(str)
        .unique()
    )

print(
    "Videojuegos con reseñas:",
    f"{len(ids_prod_res):,}"
)

Videojuegos con reseñas: 46,528


In [5]:
# Comprobación de la existencia de videojuegos en los metadatos
videojuegos = pd.read_parquet(
    ruta_metadatos
)

print(
    "Productos en los metadatos:",
    f"{len(videojuegos):,}"
)

print(
    "Identificadores únicos:",
    f"{len(videojuegos['parent_asin'].unique()):,}"
)

Productos en los metadatos: 46,533
Identificadores únicos: 46,533


In [6]:
# Comprobación unicidad de los identificadores en los metadatos
duplicados = (
    videojuegos["parent_asin"]
    .duplicated()
    .sum()
    )

nulos = (
    videojuegos["parent_asin"]
    .isna()
    .sum()
    )

print("COMPROBACIÓN DE LOS METADATOS")
print("-----------------------------")
print(f"Identificadores duplicados: {duplicados:,}")
print(f"Identificadores nulos: {nulos:,}")

COMPROBACIÓN DE LOS METADATOS
-----------------------------
Identificadores duplicados: 0
Identificadores nulos: 0


In [29]:
ids_metadatos = set(
    videojuegos["parent_asin"]
    .dropna() #Elimina los nulos
    .astype(str) #Convierte a string
    .unique() #Obtiene los identificadores únicos
)

ids_sin_resenas = ids_metadatos - ids_prod_res

numero_sin_resenas = len(ids_sin_resenas)

porcentaje_sin_resenas = (
    numero_sin_resenas
    / len(ids_metadatos)
    * 100
)

print("COBERTURA DE RESEÑAS")
print("--------------------")
print(f"Videojuegos en metadatos: {len(ids_metadatos):,}")
print(f"Videojuegos con reseñas: {len(ids_prod_res):,}")
print(f"Videojuegos sin reseñas: {numero_sin_resenas:,}")
print(
    "Porcentaje de videojuegos sin reseñas:"
    f" {porcentaje_sin_resenas:.4f} %"
)



COBERTURA DE RESEÑAS
--------------------
Videojuegos en metadatos: 46,533
Videojuegos con reseñas: 46,528
Videojuegos sin reseñas: 5
Porcentaje de videojuegos sin reseñas: 0.0107 %


In [7]:
primer_archivo = archivos_resenas[0]

primer_bloque = pd.read_parquet(primer_archivo)

print("Archivo analizado:", primer_archivo.name)
print("Número de filas:", f"{len(primer_bloque):,}")
print("Número de columnas:", primer_bloque.shape[1])

Archivo analizado: resenas_videojuegos_0000.parquet
Número de filas: 39,231
Número de columnas: 10


In [8]:
print("Columnas disponibles:")

for columna in primer_bloque.columns:
    print("-", columna)


Columnas disponibles:
- rating
- title
- text
- images
- asin
- parent_asin
- user_id
- timestamp
- helpful_vote
- verified_purchase


In [ ]:
# Comprobación de los tipos de datos de las columnas de bloques de reseñas

estructura_resenas = pd.DataFrame({
    "Columna": primer_bloque.columns,
    "Tipo de dato": primer_bloque.dtypes.astype(str).values
})

estructura_resenas

,Columna,Tipo de dato
0,rating,int64
1,title,str
2,text,str
3,images,object
4,asin,str
5,parent_asin,str
6,user_id,str
7,timestamp,datetime64[ms]
8,helpful_vote,int64
9,verified_purchase,bool


In [10]:
primer_bloque.info()

<class 'pandas.DataFrame'>
RangeIndex: 39231 entries, 0 to 39230
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   rating             39231 non-null  int64         
 1   title              39231 non-null  str           
 2   text               39231 non-null  str           
 3   images             39231 non-null  object        
 4   asin               39231 non-null  str           
 5   parent_asin        39231 non-null  str           
 6   user_id            39231 non-null  str           
 7   timestamp          39231 non-null  datetime64[ms]
 8   helpful_vote       39231 non-null  int64         
 9   verified_purchase  39231 non-null  bool          
dtypes: bool(1), datetime64[ms](1), int64(2), object(1), str(5)
memory usage: 21.9+ MB


In [11]:
columnas_principales = [
    "rating",
    "title",
    "text",
    "parent_asin",
    "timestamp",
    "helpful_vote",
    "verified_purchase"
]

columnas_existentes = [
    columna
    for columna in columnas_principales
    if columna in primer_bloque.columns
]

primer_bloque[columnas_existentes].head(10)

,rating,title,text,parent_asin,timestamp,helpful_vote,verified_purchase
0,4,It’s pretty sexual. Not my fav,I’m playing on ps5 and it’s interesting. It’s...,B07DK1H3H5,2020-12-17 06:33:24.795,0,True
1,5,Good. A bit slow,Nostalgic fun. A bit slow. I hope they don’t...,B07SRWRH5D,2020-04-16 15:31:54.941,1,False
2,5,... an order for my kids & they have really en...,This was an order for my kids & they have real...,B07MFMFW34,2017-03-30 12:37:11.000,0,True
3,5,What a great game! Fun for everyone and keeps ...,"So, I got this and used this game for the firs...",B00KTORA0K,2014-12-12 00:13:56.000,0,False
4,4,Disney Lion King 1.5,Updating my Disney movies to Blu Ray. was happ...,B071WMZ439,2017-12-27 18:12:05.463,0,True
5,5,LOVE THE UDRAW,I bought this as a Christmas present to my gra...,B003ZTTCBQ,2011-02-04 08:08:21.000,1,True
6,5,Good Chocolates,Husband bought these for me for our 25th Anniv...,B00HBUOVIO,2022-12-26 19:38:06.117,0,True
7,3,Monopoly variety game for Switch,I purchased a Switch to be able to play with m...,B07J6ZZ83K,2021-01-28 05:27:48.884,3,True
8,5,Non stop fun,I still own my Wii which works perfectly and I...,B00KTORA0K,2014-12-12 15:07:53.000,0,True
9,1,did not work for more than an hour,Stopped working an hour after my son started p...,B07X13HNK3,2013-12-31 22:11:02.000,1,True


In [26]:
columnas_referencia = list(primer_bloque.columns)

In [ ]:
# Comprobación de la consistencia de las columnas en los bloques de reseñas
for archivo in archivos_resenas:
    bloque = pd.read_parquet(archivo)
    archivos_columnas_diferentes = []

    columnas_bloque = list(bloque.columns)

    if columnas_bloque != columnas_referencia:
        archivos_columnas_diferentes.append(archivo.name)
        
print("CONSISTENCIA DE COLUMNAS")
print("-----------------------")
print(f"Archivos analizados: {len(archivos_resenas)}")
print(
    "Archivos con columnas diferentes:",
    len(archivos_columnas_diferentes)
)

if archivos_columnas_diferentes:
    print("Archivos que deben revisarse:")

    for nombre_archivo in archivos_columnas_diferentes:
        print("-", nombre_archivo)
else:
    print(
        "Todos los archivos tienen las mismas "
        "columnas y en el mismo orden."
    )

CONSISTENCIA DE COLUMNAS
-----------------------
Archivos analizados: 47
Archivos con columnas diferentes: 0
Todos los archivos tienen las mismas columnas y en el mismo orden.


In [31]:
#Comprobación de la consistencia de los tipos de datos en los bloques de reseñas
tipos_referencia = (primer_bloque.dtypes.astype(str).values)

archivos_tipos_diferentes = []

for archivo in archivos_resenas:
    bloque = pd.read_parquet(archivo)
    tipos_bloque = bloque.dtypes.astype(str).values

    if not (tipos_bloque == tipos_referencia).all():
        archivos_tipos_diferentes.append(archivo.name)
        
print("CONSISTENCIA DE TIPOS DE DATOS")
print("-----------------------------")
print(f"Archivos analizados: {len(archivos_resenas)}")
print(
    "Archivos con tipos de datos diferentes:",
    len(archivos_tipos_diferentes)
)

if archivos_tipos_diferentes:
    print("Archivos que deben revisarse:")

    for nombre_archivo in archivos_tipos_diferentes:
        print("-", nombre_archivo)
else:
    print(
        "Todos los archivos tienen los mismos "
        "tipos de datos en las columnas."
    )

CONSISTENCIA DE TIPOS DE DATOS
-----------------------------
Archivos analizados: 47
Archivos con tipos de datos diferentes: 0
Todos los archivos tienen los mismos tipos de datos en las columnas.


In [38]:
# Contar valores nulos
nulos_por_columna = pd.Series(0, index=primer_bloque.columns)
total_filas_analizadas = 0

for archivo in archivos_resenas:
    bloque = pd.read_parquet(archivo)
    nulos_bloque = bloque.isna().sum()

    nulos_por_columna += nulos_bloque
    total_filas_analizadas += len(bloque)
    
print("ANÁLISIS DE VALORES NULOS")
print("------------------------")
print(f"Total de filas analizadas: {total_filas_analizadas:,}")
print("Valores nulos por columna:")
print(nulos_por_columna)

ANÁLISIS DE VALORES NULOS
------------------------
Total de filas analizadas: 1,502,718
Valores nulos por columna:
rating               0
title                0
text                 0
images               0
asin                 0
parent_asin          0
user_id              0
timestamp            0
helpful_vote         0
verified_purchase    0
dtype: int64
